# Generic Android image → TWRP Device Tree

This notebook is the vendor-neutral path for **Samsung, Google, Xiaomi, OnePlus, Motorola, Sony, Nothing, ASUS, Fairphone and other Android devices** when you already have the relevant boot/recovery images.

Upload a dedicated `recovery.img`, or the appropriate `boot.img` / `vendor_boot.img` / `init_boot.img` set for a modern split layout. NEXT parses the images locally in Colab; it does not use Samsung FUS in this notebook.

The result is a candidate TWRP device-tree ZIP, not proof of bootability. OEM-specific signing, AVB, encryption, SELinux, display/input and image-wrapper requirements still require target validation.

In [ ]:
import os, subprocess, sys
repo = '/content/twrpdtgen-NEXT'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/RectifyData/twrpdtgen-NEXT.git', repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
os.chdir(repo)
subprocess.run(['bash', 'launcher.sh', '--setup-only'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
next_version = subprocess.check_output([sys.executable, '-c', 'import twrpdtgen; print(twrpdtgen.__version__)'], text=True).strip()
print(f'twrpdtgen-NEXT {next_version} @ {commit} ready')


## Upload images

Upload all images required for this device in one picker. For a classic device this is normally only `recovery.img`. For modern split devices, upload the primary image plus any required companions.

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
for uploaded_name, data in uploaded.items():
    safe_name = Path(uploaded_name).name
    destination = Path('/content') / safe_name
    destination.write_bytes(data)
    print(f'Uploaded {destination} ({len(data)} bytes)')


## Generation options

`primary_image` is the recovery-bearing image NEXT should inspect first. Leave companion names blank when the device does not use them. Partition-size values must be verified from GPT/lpdump/official partition metadata; leave them blank instead of guessing.

In [ ]:
primary_image = 'recovery.img' #@param {type:'string'}
boot_image = '' #@param {type:'string'}
vendor_boot_image = '' #@param {type:'string'}
init_boot_image = '' #@param {type:'string'}
skeleton = 'auto' #@param ['auto', 'omni', 'twrp-12.1', 'twrp-14.1']
recovery_layout = 'auto' #@param ['auto', 'dedicated', 'boot', 'vendor_boot']
recovery_fragment = '' #@param {type:'string'}
recovery_root_mode = 'auto' #@param ['auto', 'minimal', 'full']
boot_partition_size = '' #@param {type:'string'}
recovery_partition_size = '' #@param {type:'string'}
super_partition_size = '' #@param {type:'string'}
dynamic_group_size = '' #@param {type:'string'}
include_stock_root_in_zip = False #@param {type:'boolean'}
archive_name = 'twrp-device-tree.zip' #@param {type:'string'}


In [ ]:
from pathlib import Path
import shutil, subprocess, sys
from google.colab import files

def content_path(name):
    if not str(name).strip():
        return None
    value = Path(str(name).strip())
    if value.name != str(name).strip():
        raise ValueError('Image fields must be simple uploaded filenames, not paths')
    path = Path('/content') / value.name
    if not path.is_file():
        raise FileNotFoundError(f'Uploaded image not found: {path}')
    return path

primary = content_path(primary_image)
boot = content_path(boot_image)
vendor_boot = content_path(vendor_boot_image)
init_boot = content_path(init_boot_image)

tree_output = Path('/content/twrp-tree')
shutil.rmtree(tree_output, ignore_errors=True)
args = [
    str(primary),
    '-o', str(tree_output),
    '--skeleton', skeleton,
    '--recovery-layout', recovery_layout,
    '--recovery-root-mode', recovery_root_mode,
    '--overwrite',
]
if boot is not None:
    args.extend(['--boot-image', str(boot)])
if vendor_boot is not None:
    args.extend(['--vendor-boot', str(vendor_boot)])
if init_boot is not None:
    args.extend(['--init-boot', str(init_boot)])
if recovery_fragment.strip():
    args.extend(['--recovery-fragment', recovery_fragment.strip()])
for option, value in [
    ('--boot-size', boot_partition_size),
    ('--recovery-size', recovery_partition_size),
    ('--super-size', super_partition_size),
    ('--dynamic-group-size', dynamic_group_size),
]:
    if str(value).strip():
        args.extend([option, str(value).strip()])

cmd = [sys.executable, '-m', 'twrpdtgen', *args]
print('Running fresh vendor-neutral generation process:', ' '.join(cmd))
subprocess.run(cmd, check=True)

trees = sorted({p.parent for p in tree_output.rglob('AndroidProducts.mk')})
if len(trees) != 1:
    raise RuntimeError(f'Expected exactly one generated device tree, found {len(trees)}: {trees}')
tree = trees[0]

archive = Path('/content') / Path(archive_name).name
bundle_cmd = [
    sys.executable, '-m', 'twrpdtgen.tree_archive',
    str(tree), '-o', str(archive),
]
if include_stock_root_in_zip:
    bundle_cmd.append('--include-stock-root')
print('Bundling:', ' '.join(bundle_cmd))
subprocess.run(bundle_cmd, check=True)
print(f'Generated tree: {tree}')
print(f'Archive: {archive} ({archive.stat().st_size} bytes)')
files.download(str(archive))


## Device coverage

This notebook is deliberately OEM-neutral. The source images—not a hardcoded phone list—drive manufacturer/codename detection, architecture, boot-header geometry, recovery layout, fstab parsing and ramdisk overlay selection.

The Samsung cloud notebook remains available only as a convenience for obtaining Samsung firmware from FUS. It is not the core generator and is not required for other manufacturers.